# 多人会议语音智能解析系统
## 一、绪论
### 1.1 项目背景与意义
随着线上会议、远程办公的普及，海量会议语音数据需要人工整理，效率低下且易出错。传统语音系统仅实现语音转写，无法区分不同发言者、识别发言情绪与说话人属性。

针对该痛点，本项目设计端到端多人会议语音智能解析系统，融合声纹识别、音频多分类、语音转写技术，自动完成说话人区分、语音转文字、 情绪 / 性别 / 年龄 / 强度 识别，并生成结构化会议日记。同时针对情绪变化干扰声纹特征的行业痛点，设计情感偏移补偿策略，提升复杂情绪场景下说话人识别的鲁棒性
### 1.2 项目功能定位
结合需求与开发内容，系统核心功能分为四大模块：
- 说话人管理：支持手动注册永久说话人、自动注册临时说话人、临时说话人转正、说话人信息编辑 / 删除；
- 语音解析：支持本地音频文件解析、实时麦克风录音解析，输出时间戳 + 说话人 + 情绪 + 性别 + 年龄 + 文本结构化信息；
- 算法增强：基于 CREMA-D 数据集统计情感偏移量，对声纹特征做补偿，降低情绪对识别结果的干扰；
- 交互与导出：可视化前端界面、对话内容在线编辑、会议日记 TXT 文件导出
### 1.3 整体技术路线
整体技术栈分为数据层、算法模型层、业务逻辑层、前后端交互层：
- 数据层：TIMIT（说话人训练）、CREMA-D（情感 / 性别 / 年龄训练）；
- 特征层：FBank 梅尔频谱特征、原始语音波形；
- 模型层：ECAPA-TDNN（说话人识别）、Wav2Vec 2.0（多任务音频分类）、Whisper（语音转写）；
- 后端：Python + PyTorch + Flask；
- 前端：HTML + CSS + JavaScript + MediaRecorder（录音）；
- 优化策略：情感偏移补偿、VAD 语音活动检测、队列异步处理、声纹移动平均更新
### 1.4 报告组织结构
本文共分为九大章节：绪论、系统总体架构、实验环境与数据集、核心模块原理与代码解析、模型训练过程、系统功能流程、实验结果与分析、问题排查与优化、总结与展望

## 二、系统总体架构与数据流
### 2.1 整体框架图
```text
┌─────────────────────────────────────────────────────────────────────┐
│                           前端界面（Web）                            │
│              录音上传 / 实时录音 / 说话人注册 / 结果展示              │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    │ HTTP / WebSocket
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│                         Flask 后端 (app.py)                         │
│              路由分发 / 文件管理 / 临时会话状态 / API接口             │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│                        核心处理模块 (main.py)                        │
│                                                                     │
│   ┌─────────────┐    ┌─────────────┐    ┌─────────────┐            │
│   │  Whisper    │    │  说话人识别  │    │  Wav2Vec2   │            │
│   │  语音转文字  │───▶│  (ECAPA-    │◀───│  情感/性别/ │            │
│   │             │    │   TDNN)     │    │  年龄/强度识别 │            │
│   └─────────────┘    └─────────────┘    └─────────────┘            │
│          │                  │                  │                   │
│          └──────────────────┼──────────────────┘                   │
│                             ▼                                       │
│                    ┌─────────────────┐                             │
│                    │   结果合并与对齐  │                             │
│                    │   情感补偿（可选）│                             │
│                    └─────────────────┘                             │
└─────────────────────────────────────────────────────────────────────┘
```
![整体框架图](record/user_use.png)

### 2.2 核心业务数据流
```text
                        会议录音
                           │
                           ▼
                 ┌─────────────────┐
                 │  文件上传/实时录音 │
                 └─────────────────┘
                           │
                           ▼
                 ┌─────────────────┐
                 │  WebM→WAV转换   │   ← pydub
                 └─────────────────┘
                           │
                           ▼
         ┌─────────────────────────────────┐
         │          Whisper 转文字          │
         │  输出: segments (start, end, text) │
         └─────────────────────────────────┘
                           │
                           ▼
         ┌─────────────────────────────────┐
         │         分段音频提取              │
         │  按Whisper时间戳切分音频片段       │
         └─────────────────────────────────┘
                           │
         ┌─────────────────┼─────────────────┐
         │                 │                 │
         ▼                 ▼                 ▼
   ┌──────────┐     ┌──────────┐       ┌──────────┐
   │ ECAPA-   │     │ Wav2Vec2 │       │  临时文件 │
   │ TDNN     │     │ 情感识别  │       │  清理    │
   │ 说话人识别│     └──────────┘       └──────────┘
   └──────────┘           │
         │                │
         │   ┌────────────┘
         │   │
         ▼   ▼
   ┌─────────────────────────────────┐
   │      情感补偿（可选）             │
   │  embedding = embedding - α·bias │
   └─────────────────────────────────┘
                           │
                           ▼
                 ┌─────────────────┐
                 │   结果合并       │
                 │  相邻同说话人合并 │
                 └─────────────────┘
                           │
                           ▼
                 ┌─────────────────┐
                 │   前端展示       │
                 │ 会议日记 + 说话人列表│
                 └─────────────────┘
```
系统分为本地音频解析和实时录音解析两大使用场景，数据流略有差异
#### 2.2.1 本地音频解析数据流
1. 前端选择音频文件 → 表单上传至后端 /api/recognize
2. 后端统一转换为标准WAV格式音频
3. Whisper ASR 分割音频，输出带时间戳的文本片段
4. 逐片段处理：
   - 截取对应时间段音频 
   - 提取FBank特征
   - 情感补偿说话人识别（内部流程）
     - ECAPA-TDNN 提取声纹Embedding
     - Wav2Vec2 识别当前片段情感、性别、年龄、情感强度
     - 根据识别的情感加载情感偏移量 → 对声纹做补偿校正
     - 声纹与数据库做余弦相似度匹配 
   - 判定说话人（永久/临时/未知）
     - 匹配成功 → 返回永久说话人姓名（如"张三"）
     - 匹配失败但有历史 → 返回临时说话人编号（如"Speaker_01"）
     - 完全陌生 → 自动创建新临时说话人
5. 合并相邻同说话人片段 → 结构化结果回传前端
6. 前端渲染对话列表、说话人列表，支持编辑与导出
![本地音频解析数据流](record/local_audio.png)

#### 2.2.2 实时录音解析数据流
1. 前端请求麦克风权限 → 启动VAD音量检测（100ms/次，静音阈值600ms）
2. 检测到人声 → 新建MediaRecorder录制片段
3. 检测到超长静音 → 结束当前片段，分配序号，异步上传后端
4. 后端接收音频片段 → 执行与本地解析一致的算法流程
5. 前端维护解析队列+等待列表，严格按片段序号顺序展示结果（解决异步乱序问题）
6. 循环执行人声检测-录制-上传-解析，直至手动停止录音
7. 录音结束 →  停止VAD监测，关闭MediaRecorder，释放麦克风资源，清空后端临时说话人缓存，汇总全部会议内容，生成日记
![实时录音解析数据流](record/realtime_audio.png)


### 2.3 模块依赖关系
1. 基础预处理模块：extract_fbank.py 为 ECAPA-TDNN 提供特征输入；
2. 数据集模块：timit_dataset.py/wav2vec2_dataset.py 分别服务两大模型训练；
3. 核心模型模块：三大模型相互独立，通过main.py完成集成调度；
4. 补偿模块：compute_emotion_bias.py 预计算情感偏移量，为emotion_compensated_reco.py提供补偿依据；
5. 接口层：app.py封装所有功能接口，对接前端


## 三、实验环境与数据集
### 3.1 软硬件环境
#### 3.1.1 硬件环境
- 计算设备：NVIDIA GPU（CUDA 可用）/ 普通 CPU
- 音频设备：麦克风（实时录音使用）
- 存储：模型权重、数据集、声纹数据库本地存储
#### 3.1.2 软件环境
- 操作系统：Windows 10 / Windows 11
- 编程语言：Python 3.8+
- 核心依赖库：
  - 深度学习框架：torch、torchaudio
  - 音频处理：librosa、pydub、wave
  - 大模型：transformers（Wav2Vec2）、openai-whisper
  - 后端服务：Flask
  - 数据处理：pandas、numpy、json
  - 前端：原生 HTML/CSS/JavaScript
#### 3.1.3 全局配置
系统统一音频与模型超参数，全局配置如下：
```python
n_mels=40        # FBank梅尔滤波器数量
max_len=500      # 音频特征最大帧长
sr=16000         # 统一采样率16kHz
batch_size=32    # 训练批次大小
epochs=50        # ECAPA-TDNN训练轮数
lr=0.001         # 初始学习率
device=CUDA/CPU  # 自动选择计算设备
```

### 3.2 数据集介绍
本项目使用两个标准公开语音数据集
#### 3.2.1 TIMIT 数据集（说话人识别专用）
1. 用途：训练、验证、测试 ECAPA-TDNN 说话人识别模型；
2. 基本信息：美式英语语音数据集，包含462 名不同说话人，音频采样率 16kHz，环境纯净、信噪比高；
3. 预处理文件：preprocess_timit.py 解析路径、说话人 ID、性别，生成open_test_info.json索引文件；
4. 数据集加载类：timit_dataset.py 读取索引、提取 FBank 特征、生成标签映射。
#### 3.2.2 CREMA-D 数据集（多任务分类专用）
1. 用途：训练 Wav2Vec2 情感 / 性别 / 年龄多任务模型，同时用于计算情感偏移量；
2. 基本信息：包含 6 类情感、2 种性别、3 个年龄段，每条语音带有明确情感标签；
3. 标签定义：
   - 情感：ANG (愤怒)、DIS (厌恶)、FEA (恐惧)、HAP (开心)、SAD (悲伤)、NEU (中性)；
   - 性别：男 / 女；
   - 年龄：青年 (<35 岁)、中年 (35~55 岁)、老年 (>55 岁)；
4. 预处理文件：preprocess_cremad.py 解析文件名与人口统计信息，生成结构化 CSV 索引

### 3.3 项目目录结构(可见README.md)
```text
SoundWork/
├── app.py                      # Flask 入口
├── templates/                  # 前端页面
│   └── index.html
│
├── static/                     # 前端资源
│   ├── css/
│   └── js/
│
│  # ------------后端------------
│
├── main.py                    # 核心集成处理
│
├── Data  # 数据集目录
│   ├── CREMA-D
│   │   ├── processed/
│   │   │   └── cremad_index.csv
│   │   └── raw/
│   │       ├── AudioWAV/
│   │       └── VideoDemographics.csv
│   └── TIMIT
│
├── models                   # 模型目录
│   ├── __init__.py
│   ├── wav2vec2.py          # wav2vec2模型代码
│   ├── ecapa_tdnn_vox.py    # ECAPA-TDNN在Vox数据集上训练对应的模型代码
│   ├── ecapa_tdnn.py        # ECAPA-TDNN模型代码
│   └── whisper_asr.py       # whisper ASR模型代码
│
├── preprocessing            # 数据预处理目录
│   ├── __init__.py
│   ├── compute_emotion_bias.py # 计算情感偏置
│   ├── extract_fbank.py     # 提取FBank特征
│   ├── preprocess_cremad.py # 预处理CREMA-D数据集
│   ├── preprocess_timit.py  # 预处理TIMIT数据集
│   ├── timit_dataset.py     # TIMIT数据集
│   └── wav2vec2_dataset.py  # wav2vec2数据集
│
├── recognition              # 识别/推理模块
│   ├── __init__.py
│   ├── emotion_compensated_reco.py # 情感补偿识别模块
│   ├── speaker_reco_vox.py      # 说话人识别模块在Vox数据集
│   ├── speaker_reco.py      # 说话人识别模块
│   └── wav2vec2_reco.py      # 情感识别模块
│
├── emotion_checkpoints      # 情感识别检查点目录
│
├── speaker_checkpoints      # 说话人识别检查点目录
│   ├── speaker_db           # 已注册的说话人数据库
│   ├── best_model.pth       # 最佳模型检查点
│   ├── emotion_bias.pth     # 情感偏置计算
│   ├── pretrain.model       # ecapa-tdnn在vox数据集预训练模型
│   └── training_history.json  # 训练历史记录
│
├── tests                    # 测试目录
│   ├── test_ecapa_open.py   # 测试说话人识别模块
│   ├── test_ecapa_tdnn.py   # 测试ECAPA-TDNN模型的关闭集
│   ├── test_adaptive_vs_fixed.py   # 测试基线模型与自适应补偿强度与固定补偿的性能
│   └── test_emotion_compensation.py # 测试情感补偿识别模块
│   
├── training                 # 训练目录
│   ├── train_speaker.py     # 训练说话人识别模型
│   └── train_wav2vec2.py    # 训练情感识别模型
│
├── utils                    # 工具目录
│   ├── __init__.py
│   ├── visual.py
│   ├── visual1.py
│   └── plot_curves.py       # 绘制训练曲线图片
│ 
├── download_model.py        # wav2vec2模型下载脚本
├── .gitignore
├── config.py
├── record/                  # 报告对应的图片文件夹
├── sound.ipynb              # 报告
└── README.md
```


## 四、核心模块原理与代码解析
按数据预处理 → 三大核心模型 → 补偿算法 → 业务逻辑 → 前后端交互顺序，逐模块讲解原理与代码实现
### 4.1 数据预处理
数据预处理是语音模型训练与推理的前置基础，本项目预处理分为音频基础处理、数据集标注预处理、声学特征提取、数据集加载封装四大子模块，分别对应不同源码文件，适配 TIMIT、CREMA-D 两套数据集与不同模型的输入要求
#### 4.1.1 整体预处理流程
```text
原始音频文件
    ↓
1. 数据集标注预处理（解析文件名/附属信息，生成索引文件）
    ↓
2. 音频预处理（重采样、单声道转换、音量/分帧、长度对齐）
    ↓
3. 声学特征提取（FBank 梅尔频谱 / 原始波形，区分两大模型输入）
    ↓
4. 数据集类封装（按索引读取数据、构建标签映射、批次输出）
    ↓
模型输入
```
#### 4.1.2 数据集标注预处理
该环节主要作用：解析原始数据集文件名、配套人口信息表，生成结构化索引文件（CSV/JSON），统一管理音频路径、标签、说话人信息，避免硬编码路径，方便训练与测试。分为 TIMIT、CREMA-D 两套预处理逻辑
##### （1）CREMA-D 数据集预处理 preprocess_cremad.py
**功能说明**: 
- CREMA-D 用于 Wav2Vec2 多任务模型训练 和 情感偏移量计算，原始数据仅靠文件名编码表达情感、强度，配套 VideoDemographics.csv 存储演员性别、年龄信息。本脚本完成标签映射、信息融合、索引生成。

**标签规则设计**
1. 情感映射：
  - ANG = 愤怒 (0)、DIS = 厌恶 (1)、FEA = 恐惧 (2)、HAP = 开心 (3)、SAD = 悲伤 (4)、NEU = 中性 (5)
2. 情感强度：LO = 低强度 (0)、HI = 高强度 (1)，XX 标记为无效 (-1)
3. 性别映射：Male = 男 (0)、Female = 女 (1)
4. 年龄分段（按实际数据分布划分）：
  - 青年：年龄 < 35
  - 中年：35 ≤ 年龄 < 55
  - 老年：年龄 ≥ 55

**核心逻辑与代码解读**
1. 读取音频目录与人口统计表，遍历所有 .wav 音频文件；
2. 从文件名分割出 演员ID、情感编码、强度编码；
3. 根据演员 ID 匹配人口表，获取原始年龄、性别；
4. 执行标签数值转换，统一格式；
5. 整合「音频路径、演员 ID、情感、性别、年龄」等字段，输出 cremad_index.csv 索引文件

**关键设计点**
1. 兼容无强度标签（XX）的音频，统一标记为 -1，不参与模型训练；
2. 保留中英文情感名称双字段，兼顾模型训练（数字标签）与前端展示（文字标签）；
3. 自动创建输出目录，保证项目路径鲁棒性

##### （2）TIMIT 数据集预处理 preprocess_timit.py
**功能说明**
- TIMIT 数据集专门用于 ECAPA-TDNN 说话人模型训练与测试，原始文件按方言区、说话人分层存储，脚本负责路径解析、说话人 ID 提取、性别推断，输出 JSON 格式索引 open_test_info.json，用于开放集测试

**解析规则**
1. 路径结构：TEST/方言区/说话人文件夹/音频文件；
2. 性别推断：说话人 ID 前缀 F= 女性、M= 男性；
3. 剥离性别前缀，提取纯说话人 ID，作为模型分类标签；
4. 记录方言区、句子类型等附属信息，用于后续数据分析
   
**核心逻辑**
- 遍历 TIMIT TEST 目录下所有 WAV 文件，解析相对路径与层级信息，提取说话人、性别、句子属性，结构化存入 JSON 文件，为 timit_dataset.py 提供数据源

#### 4.1.3 音频预处理（通用规则）
项目全局统一音频标准，所有音频在进入模型前，必须完成以下统一处理，适配 16kHz 采样率要求（统一标准）：
- 采样率统一：强制重采样至 16000Hz；
- 声道统一：双声道音频取均值转为单声道；
- 长度对齐：短时音频补零、长音频截断，保证输入维度固定；
- 格式兼容：支持 .wav、前端录音 .webm 等格式，后端自动转码
该逻辑分散在特征提取、数据集类、后端接口中，是全项目通用基础规则
#### 4.1.4 声学特征提取 extract_fbank.py
##### (1)特征选择原因
相较于传统 MFCC，FBank 保留更多原始声学细节，信息损失更小，更适合深度学习模型做细粒度的音色、声纹区分，因此本项目说话人识别模块全程使用 FBank
##### (2)处理流程
原始音频波形 → 预加重 → 短时傅里叶变换 → 梅尔滤波映射 → 对数运算 → 维度对齐
##### (3)关键参数选择
- 采样率 sr=16000Hz：匹配 TIMIT 原始音频标准；
- n_mels=40：TIMIT 数据量小、环境干净，40 维滤波器足以表征特征，80 维会出现高频零值，浪费计算；
- n_fft=512：分帧长度约 32ms，符合语音处理标准（10~40ms）；
- hop_length=256：帧移 16ms；
- max_len=500：统一特征帧长，过长截断、过短补零，保证模型输入维度一致；
- 预加重系数 0.97：补偿语音高频能量衰减
##### (4) 代码核心解析
```python
def extract_fbank(file_path,n_mels=80,max_len=250,sr=16000):
    # 1. 加载音频波形
    y,sr = librosa.load(file_path, sr=sr)
    # 2. 预加重
    y=librosa.effects.preemphasis(y,coef=0.97)
    # 3. 提取梅尔谱
    mel_spec=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=n_mels,n_fft=512,hop_length=256)
    # 4. 对数FBank
    log_mel=librosa.power_to_db(mel_spec,ref=np.max)
    # 5. 长度对齐
    if log_mel.shape[1]<max_len:
        pad_len=max_len-log_mel.shape[1]
        log_mel=np.pad(log_mel,((0,0),(0,pad_len)),mode='constant')
    else:
        log_mel=log_mel[:, :max_len]
    return log_mel
```

#### 4.1.5 数据集加载封装
基于 PyTorch Dataset 类封装，将索引文件、特征提取、标签映射整合，实现训练 / 测试时的批次数据读取，分为两套专用数据集。
##### （1）TIMIT 数据集 timit_dataset.py
**功能**
- 为 ECAPA-TDNN 模型提供训练 / 测试数据，读取 JSON 索引，自动调用 extract_fbank 提取 FBank 特征，构建说话人 ID→数字标签映射字典

**核心逻辑**
1. 加载 open_test_info.json 索引，遍历所有音频条目；
2. 收集全部说话人 ID，生成唯一数字标签（模型分类输出）；
3. __getitem__ 方法：按索引读取音频 → 提取 FBank 特征 → 转为张量 → 返回特征 + 标签；
4. 适配 PyTorch DataLoader 批次加载机制

##### （2）Wav2Vec2 数据集 wav2vec2_dataset.py
**功能**
- 为 Wav2Vec2 多任务模型提供数据，直接读取原始音频波形（无需手动提取特征），适配预训练模型输入要求

**核心逻辑**
1. 读取 CREMA-D 的 CSV 索引文件；
2. 音频处理：加载波形 → 转单声道 → 重采样至 16kHz → 3 秒长度对齐；
3. 输出：原始波形张量 + 情感标签 + 性别标签 + 年龄标签 + 情感强度
4. 异常兼容：音频读取失败时返回全零张量，保证训练流程不中断

**优化**
- 废除所有基于梅尔频谱转换的繁琐代码。
- 预处理仅保留核心操作
- 音频直接化作长度为 48,000 的一维时域张量喂入模型，大幅降低了 CPU 的特征计算开销

### 4.2 说话人识别模块（ECAPA-TDNN）
#### 4.2.1 模型原理（ecapa_tdnn.py）
ECAPA-TDNN 是目前工业界主流声纹识别模型，基于TDNN（时延神经网络）+ Res2Net + SE 通道注意力 + 注意力统计池化架构，专门解决变长语音的声纹提取问题

模型整体结构：
1. 初始卷积层：一维卷积将 FBank 特征升维至 512 维，搭配 BN+ReLU 激活；
2. 三层 SE-Res2Block：核心特征提取单元，融合 Res2Net 多尺度分支与 SE 通道注意力，增强特征表征能力；
3. 维度变换卷积：映射至 1536 维特征；
4. 注意力统计池化：计算加权均值 (μ) 与标准差 (σ)，拼接得到全局语音表征；
5. 全连接层：生成 512 维归一化声纹 Embedding；
6. 分类头：训练阶段用于说话人分类，推理阶段舍弃，仅输出 Embedding
```text
输入 FBank特征 [batch, n_mels, time]
           │
           ▼
    ┌──────────────┐
    │   Conv1D     │  卷积层: 40→512
    │   BN + ReLU  │
    └──────────────┘
           │
           ▼
    ┌──────────────────────────────┐
    │      SE-Res2Block × 3         │
    │  ┌─────────────────────────┐  │
    │  │ 1×1 Conv → Split →      │  │
    │  │ 3×3 Conv (Res2Net) →    │  │
    │  │ Concat → 1×1 Conv → SE  │  │
    │  └─────────────────────────┘  │
    └──────────────────────────────┘
           │
           ▼
    ┌──────────────┐
    │    Conv1D    │  512→1536
    └──────────────┘
           │
           ▼
    ┌──────────────────────────────┐
    │      注意力统计池化 (ASP)      │
    │  attn = Sigmoid(Conv1d(x))   │
    │  μ = Σ(x·attn) / Σ(attn)     │
    │  σ = √(Σ(x²·attn)/Σ(attn) - μ²)│
    │  out = concat(μ, σ)          │
    └──────────────────────────────┘
           │
           ▼
    ┌──────────────┐
    │    BN + FC   │  1536×2 → 512
    │   L2归一化    │
    └──────────────┘
           │
           ▼
       embedding (512维)
           │
           ▼
    ┌──────────────┐
    │  Classifier  │  512 → n_classes
    └──────────────┘
```
![模型结构](record/ecapa_tdnn_arch.png)

（1）SE_Res2Block 模块
- Res2Net：将特征按通道切分为多分支（scale=8），分支间串行卷积，捕捉多尺度语音特征；
- SEBlock：通道注意力机制，自适应强化有效声学通道、抑制无效噪声通道；
- 残差连接：缓解梯度消失，提升训练稳定性

（2）注意力统计池化（ASP）
- 传统做法：全局平均池化，丢失时间维度的权重信息
- ASP改进：学习每个时间帧的重要性权重，对重要帧给予更高关注:
$$
\mu = \frac{\sum(x\cdot attn)}{\sum(attn)},\quad \sigma = \sqrt{\frac{\sum(x^2\cdot attn)}{\sum(attn)} - \mu^2}
$$
- 拼接μ(加权均值)与σ(加权标准差)作为池化输出,更好地描述embedding分布

（3）L2归一化
- 将embedding投影到单位超球面，使余弦相似度计算等价于内积

#### 4.2.2 模型前向逻辑（区分训练/推理）
```python
def forward(self,x,is_train=True):
    # 基础卷积+激活
    x=self.conv1(x);x=self.bn1(x);x=self.relu(x)
    # 三层核心残差块
    x=self.layer1(x);x=self.layer2(x);x=self.layer3(x)
    x=self.conv2(x)
    # 注意力统计池化
    attn=self.attention(x)
    mu=torch.sum(x*attn,dim=2)/torch.sum(attn,dim=2)
    sg=torch.sqrt((torch.sum((x**2)*attn,dim=2)/torch.sum(attn,dim=2))-mu**2)
    x=torch.cat((mu,sg),dim=1)
    # 生成归一化Embedding
    x=self.bn2(x);x=self.fc(x)
    embedding=F.normalize(x,p=2,dim=1)
    if not is_train:
        return embedding  # 推理：返回声纹特征
    logits=self.classifier(embedding)
    return logits       # 训练：返回分类概率
```
#### 4.2.3 说话人识别推理逻辑（speaker_reco.py）
##### 1. 注册流程
- 用户提供3-5条语音，模型提取每条语音的512维Embedding
- 取均值作为该说话人的声纹模板，存入本地.npy文件构建声纹库
##### 2. 识别流程
- 输入待识别语音片段，提取Embedding
- 与声纹库中所有模板计算余弦相似度
- 最高相似度大于阈值（0.52）→ 返回对应说话人姓名
- 否则 → 进入临时说话人机制
##### 3. 临时说话人机制
- 首次出现陌生人 → 自动生成Speaker_XX临时ID，临时说话人声纹仅缓存在内存（temp_speakers 字典中），不写入磁盘，不持久化，当前会议结束后失效
- 后续该临时说话人再次出现 → 移动平均更新声纹，不重复创建：
    emb_new=(emb_old * n + emb_cur)/(n+1)
    其中 n 为该临时说话人已出现的次数
- 用户可右键选择"注册为永久说话人"
  - 质量检查：样本数 ≥ 3 且 总时长 ≥ 5 秒方可转正
  - 注册后声纹永久保存到磁盘，跨会议可用
```text
音频片段
    │
    ▼
ECAPA-TDNN → embedding (512维)
    │
    ▼
与注册库中的embedding计算余弦相似度
    │
    │   similarity = dot(emb1, emb2) / (|emb1|·|emb2|)
    │
    ▼
max_similarity > threshold ?
    │
    ├── 是 → 返回对应说话人姓名
    └── 否 → 进入临时说话人机制
```

### 4.3 情感/性别/年龄/情感强度多任务识别模块（Wav2Vec 2.0）
#### 4.3.1 设计动机
- 在项目初期，基于 FBank + CRNN 的基线模型虽然在单任务情感识别上达到了 61.24% 的准确率，但随着任务扩展至多任务（情绪、性别、年龄），其架构天花板逐渐显现。FBank 频谱图作为人工提取特征，在傅里叶变换过程中不可避免地丢失了音频的相位信息，而这些隐蔽的物理波动往往蕴含着与情感、年龄相关的细微线索。
- 为解决这一瓶颈，本研究引入 Wav2Vec 2.0 作为语音特征提取基座。该模型在数万小时无标注语音上进行自监督对比预测编码预训练，能够直接从原始波形中学习离散声学单元，并通过 Transformer 编码器提取 768 维的高维上下文语义向量。相较于传统人工特征，Wav2Vec 2.0 能够保留更完整的声学信息，为后续多任务分类提供更丰富的特征表征
#### 4.3.2 模型原理（wav2vec2_model.py）
Wav2Vec 2.0 是 Facebook 提出的自监督语音预训练大模型，基于海量无标注语音预训练，擅长提取上下文语音特征。用于本项目做多任务微调：

模型整体结构：
1. 输入：原始 1D 语音波形（16kHz），无需手动提取特征
2. 主干网络：加载开源预训练wav2vec2-base，冻结底层 CNN 特征提取器，仅微调 Transformer 层，有效节省显存并防止灾难性遗忘
3. 池化方式：全局平均池化 (GMP) 将变长时间帧压缩为固定维度句向量
4. 多任务输出头：
   - 情绪：6 分类（愤怒、厌恶、恐惧、快乐、悲伤、中性）
   - 性别：2 分类（男、女）
   - 年龄：3 分类（青年 <35、中年 35–55、老年 ≥55）
   - 强度：3 分类（LO 低强度、MD 中强度、HI 高强度），无效标签（XX）在损失函数中通过ignore_index=-1 屏蔽
![模型结构](record/wav2vec2_arch.png)

#### 4.3.3 基线声学模型搭建
在项目初期，为快速验证多任务学习（Multi-Task Learning, MTL）拓扑结构的可行性，本研究自底向上构建了基于 **FBank + CRNN (CNN + LSTM + Attention)** 的基线语音情感识别系统。

- **声学特征工程 (Acoustic Feature Extraction)**：本实验未采用传统的 MFCC，而是提取了包含更多非线性声学原始信息的 FBank（Filterbank）梅尔频率倒谱系数，将一维时域音频转化为二维的梅尔频谱图，为后续捕捉基频与共振峰分布提供物理表征。
- **时空联合建模网络**：
  - **CNN（卷积层）**：作为底层特征提取器，利用二维卷积核在频谱图上滑动，捕捉声学能量的局部纹理突变。
  - **LSTM（长短期记忆网络）**：接收 CNN 降维后的序列特征，建模语音的时序依赖关系。
  - **Attention（注意力机制）**：作为时间帧滤波器，自动为表达情感最强烈的核心时间步分配高权重。
- **初步基线评估**：该基线模型成功打通了数据流，在单任务（情绪分类）测试中达到了 **61.24%** 的准确率，验证了底层数据预处理与特征提取架构的正确性
#### 4.3.4 多任务联合训练
##### （1）多任务联合训练的困境
当我们在网络顶层分支接入三个独立的分类头（情绪、性别、年龄）进行联合训练时，遭遇了多任务学习中典型的“跷跷板效应”与负迁移现象

由于三个任务共享底层的 CNN + LSTM 权重参数，在反向传播阶段，不同任务产生的梯度向量在共享特征空间中产生了严重的资源抢占：
1. **性别任务的信息垄断**：性别分类（2分类）难度极低，男女基频差异在频谱图上显著。其产生的极大且方向一致的梯度迅速主导了优化方向，试图将网络退化为单纯的“基频检测器”。
2. **年龄特征的多数类坍缩**：夹在中间的年龄任务（3分类）成为牺牲品。在默认的同等权重配置下，共享网络无法为其分配足够的特征容量，导致年龄准确率骤降，甚至出现模型将所有样本均预测为“中年”的崩溃现象
##### （2）解决方案：非对称梯度重加权
为解决上述瓶颈，本实验对联合损失函数进行了深度的底层数学重构。在常规的多任务训练中，往往直觉性地赋予各任务等比例的权重（如权重和为 1）。但实质上，多任务联合优化的目标函数为各子任务损失的线性组合：
$$L_{total} = w_{emo} \cdot L_{emo} + w_{gen} \cdot L_{gen} + w_{age} \cdot L_{age}$$
根据微积分的线性法则，优化器在更新底层共享参数 $\theta$ 时，其总体梯度为：
$$\nabla_{\theta} L_{total} = w_{emo}\nabla_{\theta} L_{emo} + w_{gen}\nabla_{\theta} L_{gen} + w_{age}\nabla_{\theta} L_{age}$$

**核心优化假设**：损失权重 $w$ 的数学本质并非概率分配约束，而是反向传播中的梯度缩放系数。为打破特征垄断，必须突破“权重和为 1”的限制，通过暴力干预梯度流的源头，放大困难任务（情绪、年龄）的惩罚步长，同时强力压制简单任务（性别）的梯度收敛速度

经过多轮消融实验，最终锁定的最优配比为 1.5 : 0.2 : 1.0 : 1.0

#### 4.3.5 推理封装（wav2vec2_model.py）
推理类 Wav2vec2Recognizer 封装了完整的预处理、模型推理与标签映射流程，对外提供统一的 predict(audio_path) 接口
##### 核心处理流程：
1. 音频预处理：加载原始音频波形，执行单声道转换、重采样至 16kHz、3 秒长度对齐（截断或补零），输出长度为 48000 的一维张量
2. 模型推理：将预处理后的张量传入 Wav2Vec2 四任务模型，分别获取情绪、性别、年龄、强度四个分类头的输出 logits
3. 标签映射：对四个输出分别执行 argmax 取最高概率类别，通过预定义的映射字典转换为中文标签
##### 输出标签规范：
| 属性 | 类别 |	说明 |
| --- | --- | --- |
| 情绪 | 愤怒、厌恶、恐惧、快乐、悲伤、中性 |	6 分类，对应 CREMA-D 标准标签 |
| 性别 | 男、女 |	2 分类 |
| 年龄 | 	青年、中年、老年 |	3 分类，按 CREMA-D 实际年龄分布划分（<35、35-55、≥55） |
| 强度 |	LO、MD、HI |	3 分类，分别对应低强度、中等强度、高强度；XX（自然未指定）标签在训练阶段已被梯度掩码屏蔽，推理时不输出 |


### 4.4 语音转写模块（Whisper ASR）
#### 4.4.1 原理（whisper_asr.py）
- 调用 OpenAI Whisper 预训练模型，输入音频直接输出带起止时间戳的文本片段。本项目选用base轻量版本，兼顾速度与准确率
- 优势：Whisper在转写时会自动识别句子的起止时间，天然按语义边界分割，不需要额外调参
- 核心作用：将连续语音切分为独立语义片段，为后续说话人、情感识别提供时间边界
#### 4.4.2 核心逻辑（whisper_asr.py）
```python
class WhisperASR:
    def __init__(self,model_size="base"):
        self.model=whisper.load_model(model_size)
    def transcribe(self,audio_path):
        # 输出每个片段的start/end/text
        result=self.model.transcribe(audio_path,fp16=False)
        return result["segments"]
```
#### 4.4.3 方案演进
项目初期尝试使用 Wespeaker 做说话人分割，后改为Whisper 时间戳分割，对比如下：
| 对比项 | 旧方案（Wespeaker 分割） | 新方案（Whisper时间戳分割） |
| --- | --- | --- |
| 分割边界 | 固定滑动窗口，边界不准 | 语义句子边界，精准 |
| 识别相似度 | 普遍低于 0.5 | 普遍高于 0.7 |
| 代码复杂度 | 高（大量调参） |	低（直接复用 ASR 结果） |
| 依赖 | Whisper+Wespeaker + ECAPA | Whisper+ ECAPA |
| 依赖评价 | 多模型串联，开销大 | 精简依赖，一模型二用，效率更高 |

结论：使用 Whisper 时间戳替代专用分割模型，在精度、效率、可维护性上全面提升

### 4.5 情感偏移补偿
#### 4.5.1 问题背景
同一说话人在不同情绪下，语音音色、频谱会发生偏移，导致声纹 Embedding 变化，降低说话人识别准确率。为此，借鉴语音增强中的谱减法思想，将情感视为"噪声"，设计情感偏移补偿算法，在特征域减去情感偏移
#### 4.5.2 偏移量计算（compute_emotion_bias.py）
1. 对每个演员，计算其中性语音的平均embedding
2. 对每个说话人，计算各类情感 Embedding 与中性 Embedding 的差值（偏移量）；
3. 统计所有样本的平均偏移量、标准差，保存为emotion_bias.pth文件
#### 4.5.3 固定强度补偿（早期方案）
- 在项目初期，采用固定补偿强度方案：emb_new=emb_raw - strength * emb_bias
- 通过遍历[0, 1.5]，确定最优固定强度 α = 0.7
#### 4.5.4 自适应强度补偿（当前方案）
固定强度无法区分同一情感下的不同强度表达（如“平静的快乐”与“狂喜”）。为此，引入强度自适应补偿机制，
##### 强度映射
| 强度标签 | 含义 | 补偿强度 |
| --- | --- | --- |
| LO | 低强度 | 0.2 |
| MD | 中等强度 | 0.7 |
| HI | 高强度 | 1.1 |
| XX | 未知 | 0.70 |
- 将原本固定的补偿强度改成根据不同情感强度的补偿强度
- 补偿强度被限制在[0.3, 1.2]区间，防止过度校正
- 低强度情感（LO）用小补偿（0.2），避免过度校正
- 高强度情感（HI）用大补偿（1.1），充分抵消声纹偏移
- 中等强度（MD）使用中间值（0.7）
#### 4.5.5 补偿逻辑（emotion_compensated_reco.py）
继承基础说话人识别类，重写特征提取逻辑：
1. 先通过 Wav2Vec2 识别当前音频情感
2. 读取对应情感的平均偏移量
3. 根据强度自适应调整补偿强度系数
4. 按补偿强度校正声纹向量：emb_new=emb_raw - strength * emb_bias
   - strength为补偿强度
   - 由于补偿强度适中，未产生过减，增益补偿无效，所以没有采用
5. 校正后重新归一化向量，再执行相似度匹配

### 4.6 核心集成（main.py）
MeetingDiary类是整个系统的调度中枢，串联所有算法模块：
1. 初始化：加载说话人识别器（可选情感补偿）、Wav2Vec2 情感识别器、Whisper 转写模型
2. 整体转写：调用 Whisper 一次性完成音频 → 文本转换，获得带时间戳的句子列表
3. 逐片段处理：对每个句子片段，依次执行
   - 说话人识别（含临时注册与移动平均更新）
   - 情感/性别/年龄识别（Wav2Vec2 多任务模型）
   - 情感补偿（如启用）：从 Embedding 中减去对应情感的统计偏移量
4. 后处理：合并相邻且为同一说话人的片段，输出结构化会议日记

### 4.7 后端接口与前端交互
#### 4.7.1 后端 API 设计
封装所有 HTTP 接口，包含：页面路由、音频解析、说话人注册 / 编辑 / 删除、临时说话人转正、缓存清空等接口，完成音频格式转换、跨模块调用、数据返回
| 接口 | 方法 | 功能 |
| --- | --- | --- |
| /api/recognize |	POST |	识别会议录音，返回segments |
| /api/speakers |	GET |	获取所有说话人（已注册+临时） |
| /api/speakers |	POST |	注册新说话人 |
| /api/update_speaker |	POST | 更新说话人信息（姓名/性别/年龄） |
| /api/delete_speaker |	POST |	删除已注册说话人 |
| /api/promote_temp_speaker |	POST |	将临时说话人转为永久注册 |
| /api/clear_temp_speakers |	POST |	清空临时说话人缓存 |
#### 4.7.2 数据格式约定
##### 请求格式实例（/api/recognize）：
```http
POST /api/recognize
Content-Type: multipart/form-data

audio: [音频文件]
```
##### 响应格式：
```json
{
    "success": true,
    "segments": [
        {
            "time": "00:05",
            "person": "张三 (青年·男)",
            "mood": "快乐",
            "level": "MID",
            "text": "我们对齐一下本周项目进度"
        },
        {
            "time": "00:13",
            "person": "Speaker_01 (未知·未知)",
            "mood": "中性",
            "text": "前端界面已全部开发完成"
        }
    ]
}
```
#### 4.7.3 前端（index.html + main.js + style.css）
1. 页面布局：三栏布局（说话人管理区、音频解析控制区、对话展示区）；
2. 核心功能：
   - 文件上传解析、实时麦克风录音（MediaRecorder + VAD 音量检测）；
   - 说话人列表渲染、右键菜单、双击编辑；
   - 对话文本 / 说话人在线编辑、会议日记 TXT 导出；
3. 关键技术点：
   - VAD 语音活动检测：100ms 检测一次音量，静音超过 600ms 判定片段结束；
   - 队列 + 序号机制：解决异步上传解析导致的对话顺序混乱问题；
   - 音频格式兼容：自动转换 WebM 录音文件为标准 WAV
![前端界面](record/front.png)
![前端界面](record/front1.png)
#### 4.7.4 前端交互流程
前端通过调用后端提供的 RESTful API 完成所有业务操作，核心交互流程如下:
1. 音频解析流程
  - 用户在“上传录音”区域选择本地音频文件，或通过实时录音模块生成音频
  - 点击“开始解析”按钮，前端将音频文件以 multipart/form-data 格式发送到 /api/recognize
  - 后端返回识别结果（segments），前端解析后渲染到对话展示区
  - 同时更新左侧说话人列表（显示本次会话中出现的临时说话人）
2. 说话人注册流程
  - 用户在左侧注册区填写姓名、选择性别和年龄段
  - 上传 3-5 条声纹语音（每条 6-9 秒）
  - 点击“注册入库”，前端将表单数据和音频文件通过 FormData 发送到 /api/speakers (POST)
  - 注册成功后，后端将声纹特征保存到磁盘，前端刷新左侧说话人列表
3. 说话人编辑与删除流程
  - 用户在左侧说话人列表中双击任意条目，弹出修改对话框
  - 修改姓名、性别或年龄后确认，前端调用 /api/update_speaker 同步更新后端数据库
  - 右键点击说话人条目，弹出上下文菜单：
    - 对临时说话人：显示“注册为永久说话人”，调用 /api/promote_temp_speaker
    - 对已注册说话人：显示“删除”，调用 /api/delete_speaker
4. 对话内容编辑流程
  - 用户在对话展示区双击某条对话的说话人姓名或文本内容
  - 弹出编辑框，修改后实时更新前端界面
  - 姓名修改仅影响当前会话显示，不自动同步到后端注册库（用户需通过左侧列表编辑实现永久修改）
5. 日记导出流程
  - 用户点击“导出 TXT 记录”按钮
  - 前端从 diaryBox 中读取当前结构化日记文本
  - 生成 Blob 对象并触发浏览器下载，文件名为 会议结构化日记.txt
```text
┌─────────────────────────────────────────────────────────────────────┐
│                           用户操作                                   │
└─────────────────────────────────────────────────────────────────────┘
                                  │
        ┌─────────────┬───────────┼───────────┬─────────────┐
        │             │           │           │             │
        ▼             ▼           ▼           ▼             ▼
   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐
   │ 上传文件 │   │ 实时录音 │   │注册说话人│   │编辑/删除│   │ 导出日记 │
   └─────────┘   └─────────┘   └─────────┘   └─────────┘   └─────────┘
        │             │           │           │             │
        ▼             ▼           ▼           ▼             ▼
   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐
   │选择音频 │   │VAD分段  │   │填写表单 │   │双击/右键│   │读取     │
   │文件     │   │+队列管理│   │+上传语音│   │弹出菜单 │   │diaryBox │
   └─────────┘   └─────────┘   └─────────┘   └─────────┘   └─────────┘
        │             │           │           │             │
        ▼             ▼           ▼           ▼             ▼
   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐
   │/api/    │   │/api/    │   │/api/    │   │/api/    │   │Blob     │
   │recognize│   │recognize│   │speakers │   │update/  │   │下载     │
   │(POST)   │   │(逐段)   │   │(POST)   │   │delete/  │   │.txt     │
   └─────────┘   └─────────┘   └─────────┘   │promote  │   └─────────┘
        │             │           │          └─────────┘        │
        ▼             ▼           ▼                │             │
   ┌─────────┐   ┌─────────┐   ┌─────────┐         │             │
   │渲染     │   │按序号   │   │刷新左侧 │         │             │
   │segments │   │顺序显示 │   │说话人列表│         │             │
   └─────────┘   └─────────┘   └─────────┘         │             │
        │             │           │                │             │
        └─────────────┼───────────┴────────────────┼─────────────┘
                      │                            │
                      ▼                            ▼
              ┌───────────────┐            ┌───────────────┐
              │  结构化会议日记  │◄───────────│  实时更新界面  │
              └───────────────┘            └───────────────┘
```

#### 4.7.5 实时录音
实时录音模块在前端完成，核心流程如下
1. 麦克风权限获取与录音初始化
  - 调用 getUserMedia 获取麦克风流，创建 MediaRecorder 实例
  - 同时创建 AudioContext 和 AnalyserNode 用于音量检测
2. VAD 语音活动检测
  - 每隔 100ms 检测一次音量
  - 音量 > 阈值（0.02）→ 判定为正在说话，若此前为静音状态则开始新片段
  - 音量 ≤ 阈值 → 判定为静音，若静音持续超过 600ms 则结束当前片段
3. 片段上传与序号分配
  - 每个片段结束时分配递增序号 segmentIndex
  - 将音频 Blob 上传至后端 /api/recognize 接口
  - 上传操作异步执行，不阻塞后续录音
4. 顺序显示控制
  - 后端返回结果后，存入 pendingSegments 等待列表
  - 只有当片段序号等于 nextExpectedIndex 时才显示
  - 显示后 nextExpectedIndex 自增，并继续检查下一个等待片段
  - 确保对话按实际录音顺序展示，不受网络延迟影响
5. 时间偏移计算
  - Whisper 返回的时间戳以当前片段为起点（从 0 开始）
  - 前端维护 totalOffset 变量，累加已处理片段的时长
  - 真实时间 = totalOffset + seg.start
```text
开始录音
    │
    ▼
请求麦克风权限，创建 MediaRecorder
    │
    ▼
启动 VAD 定时器（每 100ms 检测音量）
    │
    ├── 音量 > 阈值 → 正在说话
    │       │
    │       ├── 首次说话 → 创建新片段（MediaRecorder.start()）
    │       └── 更新最后说话时间
    │
    └── 音量 ≤ 阈值 → 静音
            │
            └── 如果最后说话时间距今 > 600ms
                    │
                    ▼
                结束当前片段 → 分配序号 → 异步上传 → 立即开始下一段录音
```


## 五、模型训练过程与参数设置
### 5.1 ECAPA-TDNN 说话人模型训练（train_speaker.py）
#### 5.1.1 训练配置
- 数据集：TIMIT 划分 90% 训练集、10% 验证集；
- 优化器：Adam，学习率lr=0.001；
- 损失函数：交叉熵损失（分类任务）；
- 训练轮数：50 轮；批次大小：32；
- 正则化：梯度裁剪（max_norm=1.0），防止梯度爆炸；
- 保存策略：保留验证集准确率最高的模型为best_model.pth。
#### 5.1.2 训练流程
1. 固定随机种子，保证实验可复现；
2. 加载数据集与模型，部署至 GPU/CPU；
3. 迭代训练：前向传播 → 计算损失 → 反向传播 + 参数更新；
4. 每轮结束执行验证，记录损失与准确率；
5. 保存最优模型与训练历史日志
#### 5.1.3 训练可视化
![ECAPA-TDNN 训练可视化](record/ecapa_training_curves.png)


### 5.2 Wav2Vec2 多任务模型训练（train_wav2vec2.py）
#### 5.2.1 训练配置
| 配置项 | 参数值 | 说明 |
|--------|--------|------|
| 数据集 | CREMA-D（四任务标注） | 6 情绪 × 2 性别 × 3 年龄 × 3 强度 |
| 训练/验证划分 | 8:2 | 随机划分 |
| 预训练基座 | wav2vec2-base | 380MB，冻结 CNN |
| 优化器 | Adam | 初始学习率 5e-5 |
| 学习率调度 | CosineAnnealingLR | T_max=20, eta_min=1e-6 |
| 批次大小 | 16 | 适配显存 |
| 训练轮数 | 20 | 足够收敛 |
| 多任务损失权重 | 情绪:性别:年龄:强度 = 1.5 : 0.2 : 1.0 : 1.0 | 梯度重加权 |
| 多卡并行 | DataParallel | 自动适配 |
#### 5.2.2 大模型精细化微调与动态学习率策略
在训练阶段，直接对参数量庞大的 Wav2vec 2.0 进行全量更新极易导致显存溢出（OOM）及预训练知识的灾难性遗忘。为此，本研究设计了极其严谨的微调与动态调度策略：
- **底层冻结**：通过 _freeze_parameters() 冻结了模型底层的 CNN 特征提取层，仅开放顶层的 Transformer 结构与自定义的四个多任务分类头进行参数更新。
- **联合损失再平衡**：在损失函数构建中，为了让全新的强度任务发挥最佳的辅助约束效果而不至于篡夺主任务的梯度流，本研究经过多轮微调，将多任务联合损失的动态配比锁定为黄金比例：
  $$L_{total} = 1.5 \cdot L_{emo} + 0.2 \cdot L_{gen} + 1.0 \cdot L_{age} + 1.0 \cdot L_{int}$$
- **余弦退火学习率调度**：放弃了传统的固定学习率，引入 CosineAnnealingLR 调度器。初始学习率设为极微小的5e-5以保护预训练知识，在 20 个 Epoch 内使其随着余弦曲线滑落，在微调后期降低到1.3e-6。这种“由粗到细”的微雕策略，能够引导模型参数平滑地沉淀到多任务流形的最优解谷底
#### 5.2.3 训练过程可视化与多任务收敛分析
##### 可视化
模型在 A40 服务器上进行了 20 个 Epoch 的四任务联合训练，训练过程的 Loss 收敛曲线与各项任务的验证集准确率（Accuracy）变化如下所示：
![Training and Validation Curves](record/training_curves111.png)
*图 2-1：Wav2vec 2.0 四任务联合微调的 Loss 收敛曲线（左）与准确率变化曲线（右）*
##### 实验图表深度解析：
1. **多任务均衡被完美验证（右图）**：
   - **性别（橙线）**：由于难度最低且被施加了 0.2 的抑制权重，其准确率在最初的 3 轮内迅速攀升并保持在 **99.13%** 的极高水平，未发生任何特征抢占与负迁移。
   - **年龄（绿线）**：得益于大模型的强表征能力与 1.0 的基础权重，年龄特征被成功解耦，曲线稳步攀升，最终斩获 **81.46%** 的优异成绩，彻底告别了阶段一中的“多数类坍缩”现象。
   - **情绪（蓝线）**：作为核心主任务（权重 1.5），其准确率在余弦退火调度器的微雕下跨越了行业公认的 75% 门槛，最终冲上了 **79.25%** 的历史峰值。
   - **强度（红线）**：在通过底层安全锁排除了 XX 标签的干扰后，红线展现出模型在面对纯粹的 LO/MD/HI 物理张力时的真实分类轨迹，最终稳步收敛至 **57.87%**，成功点亮了第四任务技能树。
2. **“过自信”背离现象的终极印证（左图）**：
   - 观察左侧图表可知，Train Loss 呈现完美的单调递减，最终降至 0.2150。然而，Validation Loss 在下降至第 8 轮左右后开始呈现明显的震荡反弹，并在第 20 轮挂在了 **3.2674** 的高位。
   - 一路飙升的 Validation Loss 与右图持续高位攀升并创下历史新高（平均准确率 79.43%）的 Accuracy 形成了鲜明的对比。这一现象在数学上完美印证了交叉熵损失的非线性惩罚特性：模型虽然整体分类越来越准，但由于对验证集中极少数主观性强、标注存在歧义的“困难样本”产生了过自信的错判，导致对数损失惩罚项被无限放大，引发了全局 Loss 的虚假飙升。这用无可辩驳的铁证坐实了本研究“按多任务算术平均准确率（Avg Acc）保存权重”决策的绝对正确性
#### 5.2.4 最终性能评估
为了客观评估微调后大模型的性能增益，我们将阶段一的基线模型（FBank + CRNN）与最终的四任务大模型（Wav2vec 2.0 + MTL）进行了核心指标的横向对比：
| 评测子任务 | 分类难度 | 基线(FBank + CRNN) | Wav2Vec2（最优权重） | 绝对提升幅度 | 可用性评估 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **情绪类别 (Emotion)** | 困难 (6分类) | 65.31% | **79.79%** | **+13.94%** | 突破音频单模态感知基准线，具备较高的实际应用价值。 |
| **年龄段识别 (Age)** | 中等 (3分类) | 71.20% | **82.67%** | **+10.26%** | 能够较好地穿透情绪干扰，有效提取声带老化的生物特征。 |
| **情绪强度 (Intensity)** | 困难 (3分类掩码) | 未支持 | **57.87%** | **成功引入** | 克服了多数类坍缩问题，能够较准确地反映发音的物理张力。 |
| **性别识别 (Gender)** | 简单 (2分类) | 94.86% | **99.13%** | **+4.27%** | 识别置信度极高，多任务场景下未发生负迁移。 |
| **四任务平均准确率** | - | 77.12% (三任务) | **79.43%** | **显著提升** | **各子任务性能均衡，成功实现了四个维度的声纹联合解析。** |
#### 5.2.5 核心任务能力图谱解析
最终输出的 best_wav2vec2_model.pth 在多任务联合学习方面展现出了良好的综合性能，其技术提升主要体现在以下三个层面：
1. 主任务（情绪识别 79.25%）的精度突破：在标准的音频情感 6 分类任务中，由于主观表达的模糊性，单模态识别存在较高的技术难点。本模型达到 79.25%，表明高阶语义向量配合后期的微学习率，有效提升了复杂声学边界的分类决策能力。
2. 强度分支（57.87%）的有效拟合：在利用 ignore_index 剔除自然流露（XX）样本后，57.87% 的准确率反映了模型在区分纯粹的 LO / MD / HI 物理能量阶梯时具备真实的辨识能力，而未陷入简单的盲猜策略。
3. 多任务的协同增强效应：情绪强度任务的引入并未挤占原有任务的参数容量，反而作为一种有效的正则化约束，倒逼底层网络提取出更具泛化性的特征表征，从而促使年龄（81.46%）和性别（99.13%）同步达到了实验开展以来的最高水平


### 5.3 情感偏移量计算
运行compute_emotion_bias.py，基于训练好的 ECAPA-TDNN模型提取全量 CREMA-D 声纹，统计各类情感相对中性的偏移向量，输出emotion_bias.pth


## 六、系统功能测试与实验结果分析
### 6.1 说话人识别封闭集测试
- 测试集： TIMIT测试集（924条，462个说话人）
- 结果：TIMIT测试集准确率=95.78%

### 6.2 ECAPA-TDNN 阈值优选测试（test_ecapa_open.py）
测试方案：基于 TIMIT 开放集测试，划分50 名注册说话人、118 名未注册说话人，遍历不同相似度阈值，评估两大核心指标：
- 已注册人识别率：正确匹配注册库的比例；
- 未注册人拒绝率：正确判定为陌生人的比例
#### 6.2.1 测试结果
| 阈值 | 已注册识别率 | 未注册拒绝率 | 评价 |
| --- | --- | --- | --- |
| 0.50 | 82.0% | 96.6% | 优秀 |
| 0.52 | 90.0% | 94.1% | 优秀 |
| 0.55 | 72.0% | 98.3% | 良好 |
| 0.60 | 90.0% | 86.4% | 优秀 |
| 0.65 | 52.0% | 99.2% | 较差 |
#### 6.2.2 分析
- 阈值0.5：已注册识别率高，但陌生人容易混入
- 阈值0.6：陌生人拒绝率高，但自己人认不出
- 阈值0.52：平衡点，两项指标均超过90%
#### 6.2.3 结论
综合平衡识别率与拒绝率，系统最优阈值设置为 0.52（该阈值为系统最终上线使用参数）：
- 已注册识别率：90.0%（45/50）
- 未注册拒绝率：94.1%（111/118）

### 6.3 情感补偿强度测试
为进一步提升补偿效果，本研究引入强度自适应机制，利用 Wav2Vec2 模型输出的强度标签（LO/MD/HI）动态调整补偿系数。通过网格搜索遍历 LO ∈ [0.2, 0.9]、MD ∈ [0.3, 1.1]、HI ∈ [0.7, 1.5]，以识别准确率为目标，寻找最优强度映射
#### 6.3.1 参数对比图分析：
![grouped_bars_24.png](record/grouped_bars_24.png)

图为情感补偿参数网格搜索的 24 组组合准确率对比，按强情绪补偿系数分为两组呈现。实验结果表明，参数调整对识别准确率的提升幅度有限，整体性能波动较小，最优组合准确率为 24.2%。同时，结果呈现出一定规律：弱情绪补偿系数越小、中 / 强情绪补偿系数越大，识别效果越优

#### 6.3.2 趋势分析：
![trend_contrast.png](record/trend_contrast.png)
图中展示了不同模型与补偿方案在情绪干扰场景下的说话人识别准确率对比，横轴为实验方案，纵轴为识别准确率。各方案性能表现如下：
- 无迁移学习的 VoxCeleb 预训练模型作为对比基线，准确率仅为18.9%，表明直接在该场景下使用通用模型效果不佳；
- 本项目搭建的基线模型准确率提升至20.8%，相比通用预训练模型提升了 1.9 个百分点，验证了自定义模型在目标场景下的基础适配性；
- 在基线模型基础上引入固定情感补偿方案后，准确率提升至23.4%，较基线模型提升 2.6 个百分点；
- 采用本次实验探索的自适应情感补偿方案后，准确率达到24.2%，为本次实验中的最优结果，较基线模型提升 3.4 个百分点，较 VoxCeleb 预训练模型提升 5.3 个百分点

### 6.4 端到端系统功能测试
#### 6.4.1 本地音频解析
##### 测试目标
验证系统对完整会议录音的解析能力：说话人识别、情感识别、语音转文字、时间对齐、相邻片段合并
##### 测试输入
25 秒混合音频（含已注册说话人与未注册说话人）
##### 预期结果
- 正确区分已注册 / 临时说话人
- 已注册的人被成功识别
- 临时说话人被自动编号（Speaker_01 / 02 / 03）
- 同一临时说话人跨片段被正确关联
- 相邻同说话人片段被合并
##### 实际输出
```text
[0.0s -> 4.0s] Speaker_01: She had your dark suit in greasy washwater all year.
[4.0s -> 8.0s] 李四: Don't ask me to carry an oily rag like that.
[8.0s -> 12.0s] Speaker_02: Materials. Ceramic modeling clay, red white or buff.
[12.0s -> 16.0s] Speaker_03: Dance is alternated with sung or spoken verses.
[16.0s -> 19.0s] Speaker_01: Don't ask me to carry an oily rag like that.
[19.0s -> 25.0s] 李四: She had your dark suit in greasy wash water all year.
```
简化日志输出：
```text
[0.0s-4.0s]  新说话人 Speaker_01 (自动注册)
[4.0s-8.0s]  李四 (已注册)
[8.0s-12.0s] 新说话人 Speaker_02
[12.0s-16.0s] 新说话人 Speaker_03
[16.0s-19.0s] Speaker_01 (再次出现，正确匹配)
[19.0s-25.0s] 李四(已注册的人被成功识别)
```
##### 关键验证结论
- 未注册说话人被自动分配临时 ID，不污染注册库
- 李四作为已注册的人被成功识别
- Speaker_01 在 0–4 秒与 16–19 秒被识别为同一人（未重复创建）
- 临时说话人与已注册说话人无混淆
- 合并逻辑正常工作

#### 6.4.2 实时录音测试
- VAD 分段、异步队列工作正常，录音、解析、展示互不阻塞，对话顺序无错乱
- 临时说话人自动注册，多次发言后声纹逐步优化
#### 6.4.3 辅助功能测试
| 功能 | 测试方式 | 结果 |
| --- | --- | --- |
| 说话人注册 | 上传 3–5 条声纹 | 成功注册 |
| 说话人编辑 | 双击修改姓名/性别/年龄 |	前后端同步，对话框与右侧机构化日志同步修改 |
| 临时说话人转正 | 右键 → 注册永久 |	质量检查生效 |
| 说话人删除 | 右键删除已注册 |	文件与数据库同步 |
| 对话编辑 | 双击修改文本/姓名 | 右侧结构化日志实时更新，不影响左侧数据库 |
| 日记导出 | 导出 TXT |	格式正确 |


## 七、问题排查、难点与解决方案
结合开发与测试过程中遇到的问题，分类总结如下：
### 7.1 实时录音的音频分段与顺序问题
- 问题：实时录音异步上传解析，网络延迟导致对话展示顺序错乱
- 解决方案：引入片段序号 + 解析队列 + 等待列表，强制按录音顺序展示结果，异步处理不影响界面顺序
![问题解决](record/split.png)

#### 7.1.1 方案演进
| 版本 | 方案 | 问题 |
|------|------|------|
| 版本一 | MediaRecorder，点击“结束录音”后才上传 | 用户体验差，等待时间长 |
| 版本二 | WebSocket 流式，每 30ms 发送一个包 | 每秒 33 个包，后端处理不过来，系统卡死 |
| 版本三 | 前端 VAD 分段 + 普通 HTTP 上传 | 多个数据块直接拼接导致 WebM 文件格式损坏 |
| 最终方案 | VAD 分段 + 独立上传 + 队列管理 | 解决了上述所有问题 |
#### 7.1.3 完整执行流程
异步非阻塞+队列管理 + 顺序显示
```text
用户点击"实时录音"
        ↓
1. 清空后端临时说话人缓存 (/api/clear_temp_speakers)
        ↓
2. 重置前端状态
   - 清空 parseData, chatBox, diaryBox
   - totalOffset = 0
   - segmentIndex = 0 (片段序号)
   - nextExpectedIndex = 0 (期望序号)
   - parseQueue = [] (清空队列)
   - pendingSegments = [] (清空等待列表)
        ↓
3. 请求麦克风权限 (getUserMedia)
        ↓
4. 启动 VAD (音量检测，每100ms检测一次)
        ↓
═══════════════════════════════════════════════════════════════
                        录音开始
═══════════════════════════════════════════════════════════════
        ↓
5. 用户开始说话（音量 > 阈值）
        ↓
6. 开始新片段 (startNewSegment)
   - 创建新的 MediaRecorder
   - 立即开始录音（不阻塞）
        ↓
7. 用户继续说话，录音中...
        ↓
8. 用户停止说话（静音 > 600ms）
        ↓
9. 结束当前片段 (endCurrentSegment)
   - 分配当前片段序号（如 0, 1, 2...）
   - 停止 MediaRecorder
   - 将音频块合并成 Blob
   - 【异步】调用 parseAudioBlob(formData, 序号) 加入队列
        ↓                                         ↓
   【立即返回，不等待】                    【后台并行执行】
        ↓                                         ↓
10. 用户说下一句话（VAD检测到音量）              队列处理循环 (while parseQueue.length > 0)
        ↓                                         ↓
11. 开始新片段 (startNewSegment)                取出队首片段 → 发送请求到后端
    - 创建新 MediaRecorder                           ↓
    - 立即开始录下一段                          后端处理 (/api/recognize)
        ↓                                      - 转写语音 (Whisper)
        ↓                                      - 识别说话人
        ↓                                      - 识别情感/性别/年龄
        ↓                                             ↓
        ↓                                    前端收到结果，存入 pendingSegments
        ↓                                             ↓
        ↓                                    调用 displayInOrder() 检查顺序
        ↓                                             ↓
        ↓                                    如果序号匹配期望值 → 立即显示
        ↓                                    否则 → 缓存，等待前面的完成
        ↓
    【说话2在录音中，说话1在解析中，两者同时进行】
        ↓
12. 用户停止说话（静音 > 600ms）
        ↓
13. 结束当前片段 → 【异步】上传说话2（分配序号1）
        ↓
    ... 重复步骤10-13 ...
        ↓
═══════════════════════════════════════════════════════════════
用户点击"结束"
        ↓
14. 停止录音 (stopRecording)
    - 停止音量监测
    - 停止当前录音器
    - 清理音频资源
    - 等待队列中的解析任务完成
        ↓
15. 录音结束，所有片段识别完成
```
**关键时间线**
```text
时间轴 →
────────────────────────────────────────────────────────────────────►

说话1:   [====录音====]
              ↓
           上传1(序号0) ──→ 解析1 ──→ 显示1
                      ↓
说话2:              [====录音====]
                          ↓
                       上传2(序号1) ──→ 解析2 (等待序号0完成)
                                  ↓       ↓
                             等待中...  序号0完成 → 显示2
                                          ↓
说话3:                                  [====录音====]
                                              ↓
                                           上传3(序号2) ──→ 解析3 (等待序号1完成)
                                                      ↓
                                                 序号1完成 → 显示3

```
说明：
- 上传1还没解析完，说话2已经开始录音，互不阻塞
- 如果解析2先完成，会等待解析1显示后才会显示
- 最终显示顺序始终是：说话1 → 说话2 → 说话3

#### 7.1.4 核心优势
- 录音不等待上传：endCurrentSegment 触发上传后立即返回
- 解析不阻塞录音：parseAudioBlob 是异步的，后台运行
- 重叠执行：说话2录音 + 说话1解析 = 同时进行
- 顺序保证：即使网络延迟，对话也按实际说话顺序显示
- 队列管理：多个请求自动排队，不会并发压垮服务器
- 用户体验：流畅无延迟，显示顺序正确

### 7.2 情绪干扰声纹特征
- 问题：愤怒、开心等情绪下，同一说话人声纹相似度大幅下降
- 解决方案：设计情感偏移补偿算法，
### 7.3 临时说话人管理问题
- 问题：陌生人每次发言都被判定为新用户，无法关联历史声纹
- 解决方案：
  - 设计临时说话人缓存 + 动态权重移动平均更新声纹，样本越多特征越稳定
  - 支持手动转正永久注册
- 实现：
  - 质量差的早期样本：权重低，不影响后续识别
  - 质量好的样本累积：越往后识别越准
  - 会议结束可选注册：用户决定是否保存
### 7.4 音频格式兼容问题
- 问题：前端 MediaRecorder 输出 WebM 格式，后端无法直接解析
- 解决方案：后端增加格式判断，使用 pydub 自动将 WebM 转标准 WAV

### 7.5 大模型下载网络阻断与全离线化部署
- 问题：在首次加载 Wav2Vec 2.0 预训练权重时，系统通过 transformers 库向 HuggingFace 官方服务器发起下载请求，但因网络连接超时反复失败，导致模型无法初始化
- 解决方案：
  - 镜像加速：在 Linux 系统层级配置环境变量 HF_ENDPOINT，将下载请求指向国内 HuggingFace 镜像节点，成功突破网络限制。
  - **全离线封装**：编写 download_model.py 脚本，在有网络的环境中提前将约 380MB 的基座模型完整拉取至本地 ./local_base_model 目录。该目录随项目代码一并部署，模型加载时直接从本地读取，实现“零延迟、无网秒级启动”
### 7.6 多任务梯度冲突与权重配比优化
- 问题：多任务联合训练遇到多任务学习中典型的“跷跷板效应”与负迁移现象
- 解决方案：非对称梯度重加权
#### 7.6.1 消融实验
本研究设计了一组严格的控制变量消融实验，探索不同权重配比对模型验证集准确率的影响。实验数据及现象如下表所示：
| 实验组别 | 联合 Loss 权重配比 <br> (情绪 : 性别 : 年龄) | 情绪准确率 (主) <br> `6分类` | 性别准确率 (辅) <br> `2分类` | 年龄准确率 (辅) <br> `3分类` | 综合平均准确率 <br> `Avg Acc` | 实验现象分析 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Control (对照组)** | $1.0 : 1.0 : 1.0$ | 61.24% | **97.52%** | 48.31% | 69.02% | **多数类坍缩**：性别任务极度过拟合，挤占共享空间；年龄任务彻底崩溃。 |
| **Test 1 (抑制性别)** | $1.0 : 0.2 : 1.0$ | 62.15% | 95.10% | 61.44% | 72.89% | 性别梯度受限后，年龄特征空间开始释放，准确率显著回升。 |
| **Test 2 (主次分明)** | $1.5 : 0.5 : 1.0$ | 63.80% | 96.22% | 65.18% | 75.06% | 情绪主任务得到强化，但性别特征依然存在微弱的梯度干扰。 |
| **Ours (最优比例)** | **$1.5 : 0.2 : 1.0$** | **65.31%** | 94.86% | **71.20%** | **77.12%** | **全局最优均衡**：主辅任务实现动态平衡，系统总收益最大化。 |
#### 7.6.2 结果分析
从上述实验数据可得出结论：通过采用 1.5 : 0.2 : 1.0 的非对称“黄金加权比例”，模型成功抑制了简单任务的过拟合，将节省出的网络容量转移给了复杂任务。
在此配置下，虽然性别准确率微弱下降，但换取了年龄准确率极其显著的反弹（从 48.31% 提升至 71.20%），同时带动核心的情绪识别任务提升至 65.31%。底层特征空间被成功调和，多任务梯度达成“纳什均衡”，这为本项目后续全面引入 Wav2vec 2.0 预训练大模型基座扫清了联合训练的理论与架构障碍
### 7.7 强度任务的多数类坍缩与梯度掩码
- 问题：在引入情绪强度任务后，系统遭遇了新的工程挑战：由于数据集中包含大量 XX（自然未指定）标签，模型为了走捷径快速削减全局交叉熵，将所有样本全部预测为 XX 
- 解决方案：为了打破这一僵局，且不破坏该样本在其他维度（情绪、性别、年龄）上的有效贡献，本研究摒弃了粗暴删除数据的做法，转而采取了“数据保留，梯度掩码”的解耦方案
  - **Loss 级别剔除**：在初始化强度损失函数时，配置参数 nn.CrossEntropyLoss(ignore_index=-1)在数学层面上，直接将类别 -1（XX）对应的交叉熵惩罚项与梯度流完全斩断
  - **准确率动态过滤**：在计算验证集强度准确率时，利用张量掩码（Tensor Masking）技术过滤掉所有 XX 样本，仅对真实的 LO/MD/HI 样本进行硬核分类精度统计，还原了强度特征真实的物理张力泛化水平
### 7.8 全掩码批次引发的 NaN 炸炉
#### 7.8.1 问题描述
在引入 ignore_index=-1 机制后，模型在训练到中后期时忽然触发了深度学习中毁灭性的 Train Loss: nan | Val Loss: nan 报错。整个网络的参数矩阵在瞬间被零分母毒害，彻底瘫痪
#### 7.8.2 原因分析
由于DataLoader进行的是随机 shuffle 抽样，在极端概率下，系统抓取的某个特定 Batch（大小为 16）内部的音频样本恰好全都是XX标签（即全为 -1）。当损失函数执行ignore_index=-1时，这一批次中所有样本的损失权重全部归零。这导致交叉熵公式在计算批次平均时，其分母（有效样本数）变为了 0。任意数除以零直接引发了数学意义上的非数（nan），并在反向传播中污染了全部权重
#### 7.8.3 解决方案：条件安全锁
为了从根本上杜绝该逻辑漏洞，本研究在训练与验证流中嵌入了条件判断安全锁。在计算loss_int之前对张量状态进行动态扫描
```python
# 触发安全锁
if (batch_int != 3).any():
    loss_int = criterion_int(out_int, batch_int)
else:
    loss_int = torch.tensor(0.0, device=device)
```
- 该安全锁确保：
  - 正常批次（含有有效标签）→ 正常计算损失
  - 全无效批次 → 损失置零，不参与梯度更新，不污染权重
### 7.9 验证集 Loss 与 Accuracy 背离
#### 7.9.1 问题描述
在完成架构升级并启动 Wav2vec 2.0 的微调后，系统在训练后期的日志中呈现出一个极其反直觉的现象。当训练推进到后期，终端打印出的情绪准确率（Emotion Accuracy）已经飙升至 78% 以上，各项辅助任务指标也在持续走高。然而，传统的 Model Checkpoint 机制却陷入了“死机”状态——系统并未保存这组极其优异的权重，并在日志中频繁提示 Validation Loss（验证集损失）正在持续升高
#### 7.9.2 原因分析
为破解这一迷局，本研究回归深度学习的底层评价体系，对 Accuracy与 Cross-Entropy Loss的数学本质进行了深度解构
- **Accuracy（硬指标，关注决策边界）**：准确率是离散的硬性指标，其计算依赖于 argmax 函数。只要模型预测正确类别的概率大于其他类别（例如 51% vs 49%），即判定为分类正确，准确率随之提升。这完全契合实际业务的最终诉求。
- **Cross-Entropy Loss（软指标，惩罚“过自信”）**：交叉熵损失是连续的软性评价，其标准公式为 $L = -\frac{1}{N} \sum y_i \log(\hat{y}_i)$。该公式的核心在于对数的非线性惩罚机制。

**背离的根源（过自信的误判）**：
- 在微调后期，模型对绝大多数简单样本的分类愈发完美（预测概率 $\hat{y}_i \to 0.99$，单样本 $Loss \to 0$）。然而，对于验证集中极少数的“困难/噪声样本”（如背景极其嘈杂、标注本身存在歧义的语音），模型可能会产生“极其自信的错误预测”（例如将真实标签为“中性”的音频，以 0.01 的概率预测为中性，却以 0.99 的极高置信度错判为“愤怒”）
- 此时，根据公式，该单一困难样本产生的损失将暴涨至 $-\log(0.01) \approx 4.605$。仅仅几个困难样本的极端惩罚值，就足以拉平甚至反超数百个正确样本带来的 Loss 下降，导致全局 Val Loss 出现虚假飙升
#### 7.9.3 解决方案：改用四任务平均准确率保存模型
弃用传统 if val_loss < best_loss: save() 逻辑，改用综合泛化能力更强的四任务算术平均准确率作为模型截获标准：
```python
avg_acc = (acc_emo + acc_gen + acc_age + acc_int) / 4.0
if avg_acc > best_avg_acc:
    torch.save(model.state_dict(), save_path)
```
该策略在第 20 轮成功截获了综合准确率 79.43% 的全局最优权重


## 八、系统总结
### 8.1 项目完成情况
本项目完整实现多人会议语音智能解析系统，所有预定功能全部落地：
- 完成 ECAPA-TDNN 说话人模型训练、调优，开放集综合指标优秀
- 完成 Wav2Vec2 多任务模型，实现情感、性别、年龄识别
- 提出并实现情感偏移补偿算法，有效提升复杂情绪场景识别精度
- 集成 Whisper 实现高精度语音转写
- 搭建前后端交互系统，支持本地解析、实时录音、说话人管理、日记导出等全功能
- 完成多组对照实验，确定最优阈值、最优补偿强度等关键参数

### 8.2 核心创新点
- 多模型融合架构：将声纹识别、语音分类、语音转写三大语音任务深度集成，面向会议场景落地
- 情感偏移补偿：针对性解决情绪对声纹特征的干扰，属于模型应用层优化
- 临时说话人动态管理：结合移动平均算法，实现陌生人自动识别与特征迭代
- VAD + 队列异步实时解析：兼顾实时性、流畅性与展示顺序，优化用户体验

### 8.3 不足与未来拓展方向
1. 现有不足
  - 仅支持单语种英语，未充分发挥 Whisper 多语言能力
  - 情感补偿强度为固定值，未实现自适应动态调整
  - 未支持批量音频文件批处理
  - 短时语音识别鲁棒性仍有提升空间
2. 未来拓展
  - 增加中文数据集训练，实现中英双语解析
  - 设计自适应补偿强度算法，根据情绪强度动态调整
  - 增加批量解析功能
  - 提高情绪鲁棒性
  - 优化前端 WebSocket 流式传输，进一步降低实时延迟


## 九、成员贡献
- 田静萱 2023211814 贡献率(1/3)：
  - 负责说话人识别模块的完整实现与训练。具体工作包括：
    - 从零实现 ECAPA‑TDNN 模型结构（SE‑Res2Block、注意力统计池化、L2 归一化等）
    - 完成 TIMIT 数据集上的模型训练与调优，封闭集准确率达到 95.78%
    - 设计并实现开放集说话人识别逻辑（余弦相似度匹配、动态阈值调优）
    - 实现临时说话人缓存与动态权重移动平均更新机制
    - 设计情感偏移补偿算法（固定强度与自适应强度）
    - 完成情感补偿强度的网格搜索与参数寻优
    - 编写测试脚本（开放集测试、情感补偿测试、端到端测试）
    - 完成系统后端整合（main.py、app.py）与 API 设计
    - 撰写实验部分的技术报告
    - 现场展示汇报
- 张靖 2023211974 贡献率(1/3)：
  - 负责 Wav2Vec2 多任务属性识别模块的训练与优化。具体工作包括：
    - 完成 CREMA‑D 数据集的预处理与标签解析
    - 实现 Wav2Vec2 四任务模型结构（情绪、性别、年龄、强度）
    - 设计多任务联合损失函数与梯度重加权策略（1.5 : 0.2 : 1.0 : 1.0）
    - 实现强度任务的梯度掩码（ignore_index）与 NaN 安全锁机制
    - 完成模型微调，四任务平均准确率达到 79.43%
    - 封装模型推理接口（wav2vec2_reco.py），输出中文标签
    - 计算情感偏移量（compute_emotion_bias.py），为补偿模块提供数据支持
    - PPT制作
- 丁虹豆 2023212355 贡献率(1/3)：
  - 负责前端界面设计与实时录音功能开发。具体工作包括：
    - 完成系统前端页面设计（三栏布局、响应式交互）
    - 实现文件上传解析与 Whisper ASR 集成
    - 实现实时录音模块（MediaRecorder + VAD 音量检测）
    - 设计并实现异步队列与顺序显示机制（解决网络延迟乱序）
    - 实现说话人列表渲染、右键菜单、双击编辑等交互功能
    - 完成会议日记导出功能（TXT 格式）
    - 编写 CSS 样式文件，优化用户体验


## 十、参考文献
```text
[1] Desplanques B, Thienpondt J, Demuynck K. ECAPA-TDNN: Emphasized Channel Attention, Propagation and Aggregation in TDNN Based Speaker Verification [C]. Interspeech, 2020.
[2] Baevski A, Zhou Y, Mohamed A, et al. Wav2Vec 2.0: A Framework for Self-Supervised Learning of Speech Representations [C]. NeurIPS, 2020.
[3] Radford A, Kim J W, Xu T, et al. Robust Speech Recognition via Large-Scale Weak Supervision [C]. ICASSP, 2023.
[4] 韩纪庆，张磊，郑铁然。语音信号处理 [M]. 清华大学出版社.
[5] CREMA-D: Crowd-Sourced Emotional Multimodal Actors Dataset.
[6] TIMIT Acoustic-Phonetic Continuous Speech Corpus.
[7] Cao H, Cooper D G, Keutmann M K, et al. CREMA-D: Crowd-Sourced Emotional Multimodal Actors Dataset[J]. IEEE Transactions on Affective Computing, 2014, 5(4): 377-390.
[8] Boll S F. Suppression of Acoustic Noise in Speech Using Spectral Subtraction[J]. IEEE Transactions on Acoustics, Speech, and Signal Processing, 1979, 27(2): 113-120.
[9] ITU-T. G.729 - Coding of Speech at 8 kbit/s Using Conjugate-Structure Algebraic-Code-Excited Linear Prediction (CS-ACELP)[S]. 1996.
```



